# Marine Sonar V2 — YOLOv8 Training Pipeline for Side-Scan Sonar (SSS)
### Smart India Hackathon: AI-Powered Automated Underwater Marine Debris & Anomaly Detection

This notebook provides a reproducible, end-to-end pipeline to train a YOLOv8 object detector on curated **Side-Scan Sonar (SSS)** datasets from **OpenSonarDatasets** (REMARO Network).

#### Class Taxonomy:
1. `pipeline` (Source: SubPipe SSS benchmark)
2. `derelict_fishing_gear` (Source: GhostVision sss-crab-pot-detection-ds)
3. `shipwreck` (Source: AI4Shipwrecks polygon envelope conversion)
4. `anthropogenic_anomaly` (Source: SeabedObjects-KLSG anthropogenic subset)

In [ ]:
# Step 1: Install Dependencies
!pip install -q ultralytics onnx onnxruntime albumentations pyyaml

In [ ]:
# Step 2: GPU Environment Verification
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Configure Dataset Structure & data.yaml
import yaml
from pathlib import Path

dataset_root = Path('/content/marine_sonar_v2_dataset')
dataset_root.mkdir(parents=True, exist_ok=True)

data_yaml = {
    'path': str(dataset_root),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'pipeline',
        1: 'derelict_fishing_gear',
        2: 'shipwreck',
        3: 'anthropogenic_anomaly'
    }
}

yaml_path = dataset_root / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"Config written to {yaml_path}")

In [ ]:
# Step 4: Train YOLOv8n Backbone
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    project='marine_sonar_v2',
    name='train_run',
    exist_ok=True
)

In [ ]:
# Step 5: Evaluate Validation Metrics
val_metrics = model.val()
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")
print(f"mAP50:     {val_metrics.box.map50:.4f}")
print(f"mAP50-95:  {val_metrics.box.map:.4f}")

In [ ]:
# Step 6: Export to ONNX for Production Deployment
onnx_path = model.export(format='onnx', imgsz=640, simplify=True)
print(f"Exported ONNX model to: {onnx_path}")
# Download or copy to backend/models/marine_sonar_v2.onnx